# 가천대 입학 Agentic RAG 챗봇 (중간평가)
- 근거는 제공된 5개 문서에서만 검색 (웹/외부 조회 없음, 문서 내용 추가/변조 없음: 로드 후 표 잡음 정리, 분할, 출처 표기만)
- 구조: HWP/PDF 로드 -> 분할 -> 임베딩(Chroma) + Kiwi BM25 하이브리드 검색 -> `create_agent(rag_tool)`
- **제출용**: 섹션 1~5 + `answer()`. 섹션 6(개발용 평가)은 제출 전에 삭제, API 키는 비우기.

In [ ]:
!pip install -qU langchain==1.4.7 langchain-core==1.7.0 langchain-openai==1.7.2 langchain-community==0.4.2 langchain-chroma==1.1.0 langgraph==1.2.16 chromadb==1.5.9 "langchain-hwp-hwpx-loader[distribution]==0.2.1" pdfplumber==0.11.10 rank-bm25==0.2.2 kiwipiepy==0.24.1 pandas

# 설치된 버전 확인 (위 고정 버전과 같아야 함). langchain-classic/text-splitters도 기록
from importlib.metadata import version
for _p in ["langchain", "langchain-core", "langchain-openai", "langchain-community", "langchain-chroma", "langgraph", "chromadb",
           "langchain-hwp-hwpx-loader", "pdfplumber", "rank-bm25", "kiwipiepy", "langchain-classic", "langchain-text-splitters"]:
    try: print(f"{_p}=={version(_p)}")
    except Exception: print(_p, "?")

In [ ]:
# 1) 설정 ----------------------------------------------------------------
import os, glob, time, shutil, hashlib, re
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")   # Colab 보안 비밀. 제출 시 코드에 키 없음
except Exception:
    os.environ.setdefault("OPENAI_API_KEY", "")                    # 로컬이면 직접 입력 (제출 전 반드시 비우기)

DOC_DIR = "/content/dataset"        # dataset.zip 압축 해제 폴더 (하위 폴더에 풀려도 재귀 탐색)

CFG = dict(
    chunk_size=800, overlap=100,
    k=4,                    # 벡터 검색 개수
    bm25_k=4,               # BM25 검색 개수 (0이면 BM25 미사용)
    embed_model="text-embedding-3-large",
    agent_model="gpt-5.4-mini",
    clean_hwp=True,         # HWP 표 파싱 잡음([merged:..], [rowspan=..]) 제거 (채택)
    source_prefix=True,     # 청크 앞에 [출처: 파일명 p.n] 부착 (채택)
    section_prefix=False,   # 실험: 청크 앞에 소속 전형 제목(예: 논술 전형)도 부착 (41문항: 변화 없음)
    interview_rows=True,   # 실험: 별표(★) 일정표를 '학과: 날짜 오전/오후' 줄로 풀어 ~700자씩 묶은 청크 추가 (원문 청크는 그대로 유지)
    struct_chunk=False,     # 실험: 문서 자체의 소제목(전형방법/지원자격/수능최저학력기준 등)에서 먼저 자른 뒤 재분할
    struct_chunk_size=1200, # struct_chunk 사용 시 조각 크기
    prompt_v2=False,        # 실험: 복합 질문은 항목별로 나눠 검색, 학과 지정 시 학과 행 우선, 전형 혼동 금지
    max_search=3,           # 에이전트가 검색할 수 있는 최대 횟수
    timeout=8,              # LLM 요청 1건당 제한(초). 문항 전체 20초 초과 방지
    max_retries=1,          # LLM 요청 재시도 횟수
    retry_budget=6,         # 답변 실패 시 재시도는 경과 시간이 이 값(초) 미만일 때만
)

SYSTEM_PROMPT = """당신은 가천대학교 2027학년도 입학 안내 챗봇입니다.
- 질문을 받으면 반드시 rag_tool로 제공 문서를 검색한 뒤, 검색된 문맥만 근거로 답변하세요. 사전 지식이나 추측은 금지입니다.
- 검색 결과가 질문에 답하기에 부족하거나 수시/정시/재외국민/지역의사 등 전형이 맞지 않으면, 질의어를 바꿔 다시 검색하세요(최대 {max_search}회).
- 수치, 날짜, 시각, 대상, 예외 조건을 정확히 포함하고, 질문이 묻는 것만 간결하게 답하세요.
- 근거가 끝내 없으면 '제공된 문서에서 확인할 수 없습니다'라고 답하세요.""".format(max_search=CFG["max_search"])

PROMPT_V2 = """
- 질문이 여러 항목(예: 모집인원, 전형방법, 지원자격)을 묻거나 여러 전형·학과·날짜를 묻는 경우, 항목별로 나누어 rag_tool을 여러 번(가능하면 한 번에 동시에, 최대 4개) 호출하고 모든 항목에 답하세요. 질의어마다 전형 이름과 항목 이름(예: '학생부우수자 전형 지원자격')을 함께 넣으세요.
- '모두', '각각', '언제', '며칠' 같은 질문은 검색된 표와 목록에서 해당하는 행을 빠짐없이 나열하세요.
- 특정 학과나 모집단위를 묻는 질문은 그 학과가 직접 적힌 표의 행을 근거로 답하고, 전형 전체 일정만으로 답하지 마세요.
- 질문의 전형(수시, 정시, 재외국민, 지역의사, 논술, 학생부우수자, 지역균형 등)과 같은 전형의 근거만 사용하세요. 다른 전형의 같은 형식 표를 섞지 마세요.
- '확인할 수 없습니다'라고 답하기 전에, 전형 이름과 항목 이름(지원자격, 전형일정 등)으로 질의어를 바꿔 한 번 더 검색하세요."""
if CFG["prompt_v2"]:
    SYSTEM_PROMPT += PROMPT_V2

TOOL_DESC = ("가천대학교 2027학년도 입학 자료(재외국민 모집요강, 지역의사선발전형, 수시 모집요강, 정시 모집요강, 수시 면접고사 일정)를 검색합니다. "
             "모집인원, 지원자격, 전형 방법, 일정, 원서접수, 전형료, 제출서류, 수능 반영, 면접 등 입학 관련 모든 질문에 사용하세요. "
             "질의어에는 전형 구분(수시/정시/재외국민/지역의사)과 핵심 키워드를 포함하세요.")

In [ ]:
# 2) 문서 로드 (HWP + PDF) ---------------------------------------------------
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_hwp_hwpx import HwpHwpxLoader

def load_docs():
    docs = []
    files = [p for p in sorted(glob.glob(f"{DOC_DIR}/**/*", recursive=True)) if os.path.isfile(p)]
    for p in files:
        name, low = os.path.basename(p), p.lower()
        try:
            if low.endswith(".hwp"):
                # 배포용 HWP는 pyhwp 백엔드가 필요
                part = HwpHwpxLoader(p, mode="elements", distribution_backend="pyhwp").load()
            elif low.endswith(".pdf"):
                part = PDFPlumberLoader(p).load()
            else:
                continue
        except Exception as e:
            print("!!! 로드 실패:", name, repr(e))
            continue
        for d in part:
            d.metadata["source"] = name        # 출처 파일명으로 통일
        docs += part
        print(name, len(part))
    return docs

docs = load_docs()
_n_src = len({d.metadata["source"] for d in docs})
if _n_src != 5:
    print(f"!!! 경고: 로드된 문서가 {_n_src}개입니다(기대 5개). DOC_DIR 경로와 HWP 로더 설치를 확인하세요.")

# 로드 후 전처리: 표 셀 병합 표기 잡음만 제거 (내용 추가/변조 없음)
MERGED_RE = re.compile(r"\[merged:R\d+C\d+\]")
SPAN_RE = re.compile(r"\s*\[rowspan=\d+, colspan=\d+\]")
def clean_hwp_text(t):
    t = SPAN_RE.sub("", MERGED_RE.sub("", t))
    t = re.sub(r"\|(?:[ \t]*\|)+", "|", t)           # 비어버린 셀 연속 제거
    t = re.sub(r"^\|[ \t]*(?:---[ \t]*\|[ \t]*)+$", "", t, flags=re.M)  # 구분선 제거
    return re.sub(r"\n{2,}", "\n", t)
if CFG["clean_hwp"]:
    for d in docs:
        if d.metadata["source"].lower().endswith(".hwp"):
            d.page_content = clean_hwp_text(d.page_content)

# 별표(★) 일정표 풀어쓰기: 표의 헤더 값과 ★ 표시만 사용하고 문서에 없는 문구는 추가하지 않음 (쪽 텍스트로는 어느 날짜 열인지 알 수 없기 때문)
import pdfplumber
from langchain_core.documents import Document

ROWS_MAX_CHARS = 700

def _header_rows(t):
    n = 0                                                        # 첫 칸이 '모집단위' 또는 빈 칸인 행 = 헤더 행
    while n < len(t) and (t[n][0] or "").strip() in ("", "모집단위"):
        n += 1
    return n

def _is_star_matrix(t):
    if len(t) < 4 or len(t[0]) < 4 or (t[0][0] or "").strip() != "모집단위":
        return False
    cells = [(c or "").strip() for r in t[_header_rows(t):] for c in r[1:]]
    return bool(cells) and all(c in ("", "★") for c in cells) and "★" in cells

def serialize_star_tables(path, name):
    out = []
    with pdfplumber.open(path) as pdf:
        for pi, pg in enumerate(pdf.pages):
            tables = [t for t in pg.extract_tables() if _is_star_matrix(t)]
            if not tables:
                continue
            title = (pg.extract_text() or "").split("\n")[0].strip()      # 쪽 첫 줄(문서 자체의 소제목)
            for t in tables:
                n_hdr = _header_rows(t)
                labels = [[] for _ in t[0]]
                for r in t[:n_hdr]:
                    last = ""
                    for j in range(1, len(r)):
                        v = (r[j] or "").strip()
                        last = v or last                                 # 병합된 머리칸은 오른쪽 열로 이어받기
                        if last: labels[j].append(last)
                date_line = " ".join(dict.fromkeys(v for r in t[:n_hdr] for v in ((c or "").strip() for c in r[1:])
                                                   if v and v not in ("오전", "오후")))      # 표 머리의 날짜/구분 값 그대로 (★ 없는 날짜도 보존)
                lines = []
                for r in t[n_hdr:]:
                    dept = (r[0] or "").replace("\n", " ").strip()
                    marks = [" ".join(labels[j]) for j in range(1, len(r)) if (r[j] or "").strip() == "★"]
                    if dept and marks:
                        lines.append(f"{dept} ★: " + ", ".join(marks))
                cur = []                                                 # 행을 한 줄씩 쪼개지 않고 ~700자씩 묶음 (원본 표 청크가 밀려나지 않게)
                for line in lines + [None]:
                    if cur and (line is None or len(title) + len(date_line) + sum(len(x) + 1 for x in cur) + len(line) > ROWS_MAX_CHARS):
                        out.append(Document(page_content=title + "\n" + date_line + "\n" + "\n".join(cur), metadata={"source": name, "page": pi, "kind": "rows"}))
                        cur = []
                    if line is not None:
                        cur.append(line)
    return out

if CFG["interview_rows"]:
    _n = 0
    for _p in sorted(glob.glob(f"{DOC_DIR}/**/*.pdf", recursive=True)):
        _extra = serialize_star_tables(_p, os.path.basename(_p))
        docs += _extra; _n += len(_extra)
    print("별표 일정표 행 추가:", _n)

In [ ]:
# 3) 분할 + 임베딩(캐시) + Chroma + Kiwi BM25 -----------------------------------
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from kiwipiepy import Kiwi

# 문서 자체의 "OO 전형" 제목 줄을 읽어, 제목이 안 보이는 뒤쪽 청크에도 소속 전형을 알려줌 (문서에 없는 내용 추가 없음)
SEC_RE = re.compile(r"^([가-힣][가-힣()·A-Za-z0-9]*(?: [가-힣()·A-Za-z0-9]+)?) 전형$", re.M)
SEC_CARRY = 3     # 제목 쪽 이후 몇 쪽까지 같은 전형으로 이어받을지
def page_titles(text):
    ms = [(m.start(), m.group(1)) for m in SEC_RE.finditer(text)]
    return [] if len(ms) >= 4 else ms            # 목차 쪽(제목이 여러 개 나열)은 제외

sec_info = {}
for src in sorted({d.metadata["source"] for d in docs}):
    pages = sorted((d for d in docs if d.metadata["source"] == src and d.metadata.get("page") is not None and d.metadata.get("kind") != "rows"),
                   key=lambda d: d.metadata["page"])
    cur, age = None, 0
    for d in pages:
        t = page_titles(d.page_content)
        sec_info[(src, d.metadata["page"])] = (t, cur if (cur and age < SEC_CARRY) else None)
        if t: cur, age = t[-1][1], 0
        else: age += 1

def section_of(s):
    t, inherited = sec_info.get((s.metadata["source"], s.metadata.get("page")), ([], None))
    start = s.metadata.get("start_index", 0)
    before = [name for pos, name in t if pos <= start]
    inside = [name for pos, name in t if start < pos < start + len(s.page_content)]
    return before[-1] if before else (inside[0] if inside else inherited)

splitter = RecursiveCharacterTextSplitter(chunk_size=CFG["chunk_size"], chunk_overlap=CFG["overlap"],
                                          separators=["\n\n", "\n", " ", ""], add_start_index=True)
# 구조 기반 분할: 문서 자체의 소제목 줄에서 먼저 자름 (소제목 줄 = 쪽 텍스트에 그대로 있는 줄)
SUBHEADS = ["전형방법", "지원자격", "전형일정", "수능최저학력기준", "선발원칙", "제출서류"]
HEAD_RE = re.compile(r"^(" + "|".join(SUBHEADS) + r")$", re.M)
struct_splitter = RecursiveCharacterTextSplitter(chunk_size=CFG["struct_chunk_size"], chunk_overlap=CFG["overlap"],
                                                 separators=["\n\n", "\n", " ", ""], add_start_index=True)

def split_docs(docs):
    if not CFG["struct_chunk"]:
        return splitter.split_documents(docs)
    out = []
    for d in docs:
        text = d.page_content
        cuts = [(m.start(), m.group(1)) for m in HEAD_RE.finditer(text)]
        bounds = ([(0, None)] if (not cuts or cuts[0][0] > 0) else []) + cuts
        for i, (a, head) in enumerate(bounds):
            b = bounds[i + 1][0] if i + 1 < len(bounds) else len(text)
            if not text[a:b].strip():
                continue
            for c in struct_splitter.create_documents([text[a:b]], metadatas=[dict(d.metadata)]):
                c.metadata["start_index"] = a + c.metadata.get("start_index", 0)
                c.metadata["subhead"] = head
                out.append(c)
    return out

splits = split_docs(docs)
if CFG["source_prefix"] or CFG["section_prefix"] or CFG["struct_chunk"]:
    for s in splits:
        p = s.metadata.get("page")
        tag = s.metadata["source"] + ("" if p is None else " p." + str(p + 1))
        if CFG["section_prefix"] or CFG["struct_chunk"]:
            sec = section_of(s)
            if sec: tag += " | " + sec + " 전형"
        if s.metadata.get("subhead"):
            tag += " > " + s.metadata["subhead"]
        s.page_content = "[출처: " + tag + "]\n" + s.page_content
print("chunks:", len(splits))

embedder = CacheBackedEmbeddings.from_bytes_store(OpenAIEmbeddings(model=CFG["embed_model"]), LocalFileStore("./cache/"))
CHROMA_PATH = "./chroma_db"
shutil.rmtree(CHROMA_PATH, ignore_errors=True)       # 재실행 시 중복 적재 방지
db = Chroma.from_documents(splits, embedder, persist_directory=CHROMA_PATH, collection_metadata={"hnsw:space": "cosine"})
vec_retriever = db.as_retriever(search_kwargs={"k": CFG["k"]})

kiwi = Kiwi()
ALLOWED_TAGS = {"NNG", "NNP", "NNB", "VV", "VA", "XR", "SL", "SN"}
def korean_tokenizer(text): return [t.form for t in kiwi.tokenize(text) if t.tag in ALLOWED_TAGS]
bm25_retriever = BM25Retriever.from_documents(splits, preprocess_func=korean_tokenizer)
bm25_retriever.k = CFG["bm25_k"]

In [ ]:
# 4) 검색 도구 + Agentic RAG -----------------------------------------------------
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
try:
    from langgraph.errors import GraphRecursionError
except Exception:
    GraphRecursionError = RecursionError

def hybrid_retrieve(query):
    docs = vec_retriever.invoke(query)
    if CFG["bm25_k"] > 0:
        docs = bm25_retriever.invoke(query) + docs
    seen, uniq = set(), []
    for d in docs:
        key = d.page_content.strip()
        if key not in seen:
            seen.add(key); uniq.append(d)
    return uniq

@tool("rag_tool", description=TOOL_DESC)
def rag_tool(query: str) -> str:
    docs = hybrid_retrieve(query)
    if not docs:
        return "<문맥 정보>검색 결과가 없습니다.</문맥 정보>"
    def label(d):
        if CFG["source_prefix"] or CFG["section_prefix"] or CFG["struct_chunk"]:      # 청크에 이미 출처가 붙어 있음
            return d.page_content
        p = d.metadata.get("page")
        return "[출처: " + d.metadata["source"] + ("" if p is None else " p." + str(p + 1)) + "]\n" + d.page_content
    return "<문맥 정보>" + "\n\n".join(label(d) for d in docs) + "</문맥 정보>"

llm = init_chat_model(CFG["agent_model"], timeout=CFG["timeout"], max_retries=CFG["max_retries"])
agent = create_agent(model=llm, tools=[rag_tool], system_prompt=SYSTEM_PROMPT)

NO_ANSWER = "제공된 문서에서 확인할 수 없습니다."
FAILS = []      # (질문, 오류) 기록: 평가 시 몇 건이 오류 경로를 탔는지 확인용

def _text(msg):
    c = msg.content
    if isinstance(c, list):                    # 일부 모델은 content를 블록 리스트로 반환
        return "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in c)
    return c

def _fallback(question):
    """에이전트가 실패했을 때: 검색 1회 + 단일 생성 (여전히 제공 문서 근거만 사용)"""
    try:
        ctx = rag_tool.invoke({"query": question})
        r = llm.invoke([("system", SYSTEM_PROMPT), ("user", ctx + "\n\n질문: " + question)])
        return _text(r)
    except Exception as e:
        FAILS.append((question, "fallback: " + repr(e)))
        return NO_ANSWER

def answer(question: str) -> str:
    """평가용 진입점: 질문 -> 답변 문자열. 오류/재귀한도/지연이 나도 예외 없이 문자열을 반환."""
    t0 = time.perf_counter()
    for attempt in range(2):
        try:
            res = agent.invoke({"messages": [{"role": "user", "content": question}]},
                               config={"recursion_limit": 2 * CFG["max_search"] + 3})
            return _text(res["messages"][-1])
        except Exception as e:
            FAILS.append((question, repr(e)))
            if isinstance(e, GraphRecursionError) or time.perf_counter() - t0 > CFG["retry_budget"]:
                break
    return _fallback(question) if time.perf_counter() - t0 < CFG["timeout"] else NO_ANSWER

In [ ]:
# 5) 동작 확인 (연결/워밍업용 일반 문장. 평가 질문이나 정답과 무관) ----------------------
t = time.perf_counter()
print(answer("안녕하세요. 입학 관련 질문에 답해 줄 수 있나요?"))
print(f"{time.perf_counter()-t:.1f}s")

---
## 6) 개발용 평가 (제출 전 이 아래 셀 전부 삭제)
`/content/dev_questions.csv`(컬럼: question, answer)를 올려 사용. 채점 공식은 README 그대로: 정답 1.6 + 시간점수(5초 이하 0.4, 초과 1초당 -0.08, 20초 초과는 0점).

In [ ]:
import pandas as pd, numpy as np, json
from langchain.messages import HumanMessage

DEV_Q = "/content/dev_questions.csv"
RESULTS = "/content/results.csv"
judge_llm = init_chat_model("gpt-5.4-mini")

def judge(q, gold, pred):
    """반환: (정답 여부, 판정 근거 한 줄). 표현 차이는 무시하고 핵심 정보 일치 여부만 본다."""
    p = ("당신은 채점관입니다. 질문, 예상 정답, 모델 답변이 주어집니다.\n"
         "- 모델 답변이 예상 정답의 핵심 정보(수치, 날짜, 시각, 대상, 조건, 예외)를 모두 담고 있고 서로 모순되지 않으면 정답입니다.\n"
         "- 표현 방식의 차이(연도 표기 유무, '오전, 오후'와 '오전과 오후', 날짜 형식, 나열 순서, 말투)는 무시합니다.\n"
         "- 예상 정답에 없는 추가 설명은, 정답과 모순되지 않으면 감점하지 않습니다.\n"
         "- 핵심 정보가 하나라도 빠지거나 다르거나, '확인할 수 없다'고 답했으면 오답입니다.\n\n"
         f"질문: {q}\n예상 정답: {gold}\n모델 답변: {pred}\n\n"
         "첫 줄에 판정 근거를 한 문장으로 쓰고, 마지막 줄에는 '판정: 정답' 또는 '판정: 오답'만 쓰세요.")
    out = judge_llm.invoke([HumanMessage(p)]).content
    out = out if isinstance(out, str) else "".join(x.get("text", "") if isinstance(x, dict) else str(x) for x in out)
    lines = [l.strip() for l in out.strip().splitlines() if l.strip()]
    last = lines[-1] if lines else ""
    return ("정답" in last and "오답" not in last), (lines[0][:200] if lines else "")

def time_score(t, ok):
    if not ok or t > 20: return 0.0
    return round(1.6 + max(0.0, 0.4 - 0.08 * max(0.0, t - 5)), 3)

LAST = None   # 질문별 요약 (ok_rate = 반복 중 정답 비율)
RUNS = None   # 모든 실행 원본
def run_eval(name, note="", repeats=1):
    """현재 CFG/agent로 질문 전체를 repeats번 실행. 시간은 answer() 호출 구간만 측정.
    repeats>=2면 정답이 실행마다 뒤집히는 문항(flaky)을 따로 보여줌 -> 1회 측정의 잡음 확인용."""
    global LAST, RUNS
    FAILS.clear()
    qs = pd.read_csv(DEV_Q); rows = []
    for rep in range(repeats):
        for qi, r in qs.iterrows():
            t = time.perf_counter()
            pred = answer(r.question)
            dt = time.perf_counter() - t
            ok, why = judge(r.question, r.answer, pred)
            rows.append(dict(rep=rep, qi=qi, q=r.question, gold=r.answer, pred=pred, why=why, ok=ok, sec=round(dt, 1), score=time_score(dt, ok)))
    RUNS = pd.DataFrame(rows)
    LAST = (RUNS.groupby("qi").agg(q=("q", "first"), gold=("gold", "first"), pred=("pred", "last"), why=("why", "last"),
                                   ok_rate=("ok", "mean"), sec=("sec", "mean")).reset_index())
    s = dict(name=name, repeats=repeats, acc=round(RUNS.ok.mean(), 3), avg_sec=round(RUNS.sec.mean(), 2),
             p90_sec=round(RUNS.sec.quantile(.9), 1), max_sec=RUNS.sec.max(),
             score_per_q=round(RUNS.score.mean(), 3), est_50q=round(RUNS.score.mean() * 50, 1), n=len(qs),
             always_wrong=int((LAST.ok_rate == 0).sum()), flaky=int(((LAST.ok_rate > 0) & (LAST.ok_rate < 1)).sum()),
             fails=len(FAILS), note=note,
             **{k: CFG[k] for k in ("chunk_size", "overlap", "k", "bm25_k", "agent_model", "embed_model", "clean_hwp", "source_prefix", "section_prefix", "interview_rows", "struct_chunk", "prompt_v2")})
    pd.DataFrame([s]).to_csv(RESULTS, mode="a", header=not os.path.exists(RESULTS), index=False)
    print(s)
    if FAILS: print("오류 경로를 탄 문항:", FAILS[:5])
    return LAST

In [ ]:
# 실험 방법: 설정(CFG)을 한 항목만 바꾸고 2)~4) 셀을 다시 실행한 뒤 run_eval 실행
# 1회 측정은 문항이 1~4개씩 뒤집히는 잡음이 있으므로 repeats=2 권장 (always_wrong=항상 틀림, flaky=왔다갔다)
df = run_eval("run1", repeats=2)

In [ ]:
# 결과 비교 / 오답 확인 (항상 틀림 = ok_rate 0, 왔다갔다 = 0<ok_rate<1)
display(pd.read_csv(RESULTS))
display(LAST.sort_values("sec", ascending=False).head(5)[["q", "sec"]])
for _, r in RUNS[~RUNS.ok].sort_values(["qi", "rep"]).iterrows():      # 틀린 실행의 전체 답변과 판정 근거
    print(f"[#{r.qi} rep{r.rep}] {r.q}\n  정답: {r.gold}\n  답변: {r.pred}\n  판정: {r.why}\n".replace("\n", chr(10)))
